In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [6]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [7]:
messages = []
add_user_message(
    messages,
    """
    What is the best exercise to reduce back pain?
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_013mYBRCWVLCCkLR4jJQuPLf', container=None, content=[TextBlock(citations=None, text="I'll search for current information on the best exercises for reducing back pain.", type='text'), ServerToolUseBlock(id='srvtoolu_016QZ5UmcmfkChbThxPuugjU', caller=None, input={'query': 'best exercises reduce back pain'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='Et4lCioIDxgCIiRhYWJlZjIzNy1mZjcyLTQzOGItOGI2MS00NThiOGYwNGIzOTYSDLJZqKjSgpDVhWnF+hoMjEm68CYQZRmsQWKKIjCob4nvgoSDMBpqzoyfCTMmNjaWWrhRQxdCbyqXV8/Xg4VzPcl56WZF4+COSYhRGH8q4SRuZruvou3i4NxmZ958SPYDN8r2lMO+DcO8B3iy7KJPKJCTfHOBDIBQLRHNeUxRFTxymRZlKDWv1ujFh/+Xu1+foEIZ5A4n6dNoPI3DCqhZj/0LLA20U1NSoJpe8kkG3zAi0ZCBwQJxahBnsSsx4gEALhcFWFEWBORfqphBlgiRZSvyISwGiszX/lnmaHe2Q2k87H8WtDVzCElvLrtDVICpRc0u/2+Lhp47LarPo7BZK43etcgZDG3D/+MmQnMCoum0NUT+0cWrC45HGz+LPeOSn6hIqvCwzKe6CAqV/S6+bqbxWZhp0luj+CK6ie/I4u8eqBYbibRPfpCauHw019kP2MJUUiHY18i9